# Assignment 1 — NLP & Computer Vision (PGCP-AI Aug 2026)
**Submission Date:** 09/11/2026  
**Student:** abs6187 (23f2000876@ds.study.iitm.ac.in)

---

## Part A: Machine Translation as an Application of NLP
**Objective:** Understand Machine Translation as an important application of NLP and implement a simple translation system using Python.

In [ ]:
# Install dependencies (run once)
# !pip install deep-translator googletrans==4.0.0-rc.1 nltk spacy textblob

In [ ]:
# --- Imports ---
from deep_translator import GoogleTranslator
import warnings
warnings.filterwarnings('ignore')

### A1. Basic Translation Demo
Using `deep-translator` to translate between languages.

In [ ]:
# English to Hindi
text_en = "Natural Language Processing helps computers understand human language."
translated_hi = GoogleTranslator(source='en', target='hi').translate(text_en)
print(f"English:  {text_en}")
print(f"Hindi:    {translated_hi}")
print()

# Hindi back to English (round-trip)
back_to_en = GoogleTranslator(source='hi', target='en').translate(translated_hi)
print(f"Back to English: {back_to_en}")

### A2. Multi-language Translation

In [ ]:
source_text = "Machine translation bridges the language barrier between people."
target_langs = {'hi': 'Hindi', 'mr': 'Marathi', 'ta': 'Tamil', 'fr': 'French', 'de': 'German'}

print(f"Source (English): {source_text}\n")
for code, name in target_langs.items():
    result = GoogleTranslator(source='en', target=code).translate(source_text)
    print(f"{name:>10}: {result}")

### A3. Translation Quality — BLEU Score (Optional Analysis)

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize

# Reference translation (human)
reference = "Natural Language Processing helps computers understand human language."

# Round-trip translation
hindi = GoogleTranslator(source='en', target='hi').translate(reference)
round_trip = GoogleTranslator(source='hi', target='en').translate(hindi)

ref_tokens = word_tokenize(reference.lower())
hyp_tokens = word_tokenize(round_trip.lower())

score = sentence_bleu([ref_tokens], hyp_tokens)
print(f"Original:    {reference}")
print(f"Round-trip:  {round_trip}")
print(f"BLEU Score:  {score:.4f}")

---

## Part B: Chatbot Architecture — Rule-Based NLU + NLG
**Objective:** Develop a simple rule-based chatbot that performs Natural Language Understanding (NLU) and Natural Language Generation (NLG).

### B1. Architecture Overview

```
User Input
    │
    ▼
┌─────────────────┐
│  NLU Module      │  ← Intent Detection + Slot Extraction
│  (understand)    │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Dialogue Mgr    │  ← Decides response strategy
│  (manage)        │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  NLG Module      │  ← Generates natural language reply
│  (generate)      │
└─────────────────┘
         │
         ▼
    Bot Response
```

In [ ]:
import re


# ────────────── NLU Module ──────────────
def understand(text: str) -> dict:
    """Natural Language Understanding: extract intent and slots from user input."""
    text_lower = text.lower().strip()

    # Intent detection using keyword patterns
    intent_patterns = {
        'greeting':    r'\b(hello|hi|hey|good\s*(morning|evening|afternoon))\b',
        'farewell':    r'\b(bye|goodbye|see\s*you|exit|quit)\b',
        'book_flight': r'\b(book|fly|flight|travel|ticket)\b',
        'weather':     r'\b(weather|temperature|rain|forecast)\b',
        'help':        r'\b(help|assist|support|what\s*can)\b',
    }

    intent = 'unknown'
    for name, pattern in intent_patterns.items():
        if re.search(pattern, text_lower):
            intent = name
            break

    # Slot extraction
    slots = {}
    loc_matches = re.findall(r'\b(?:from|to|in|at)\s+(\w+)', text_lower)
    if loc_matches:
        slots['locations'] = [loc.title() for loc in loc_matches]

    from_match = re.search(r'from\s+(\w+)', text_lower)
    to_match = re.search(r'to\s+(\w+)', text_lower)
    if from_match:
        slots['origin'] = from_match.group(1).title()
    if to_match:
        slots['destination'] = to_match.group(1).title()

    return {'intent': intent, 'slots': slots, 'raw': text}


# ────────────── NLG Module ──────────────
def generate(nlu_result: dict) -> str:
    """Natural Language Generation: produce a response based on NLU output."""
    intent = nlu_result['intent']
    slots = nlu_result['slots']

    responses = {
        'greeting':  'Hello! I can help you book flights or check the weather. What would you like?',
        'farewell':  'Goodbye! Have a great day! ✈️',
        'help':      'I can help with:\n  • Booking flights (say "book a flight from X to Y")\n  • Weather info (say "weather in X")',
        'unknown':   "Sorry, I didn't understand that. Type 'help' to see what I can do.",
    }

    if intent == 'book_flight':
        origin = slots.get('origin', '?')
        dest = slots.get('destination', '?')
        if origin != '?' and dest != '?':
            return f'🔍 Searching flights from {origin} to {dest}... Found! Best fare: ₹4,500.'
        elif origin != '?':
            return f'Flying from {origin} — where to?'
        else:
            return 'Sure! Where are you flying from and to?'

    if intent == 'weather':
        locs = slots.get('locations', [])
        if locs:
            return f'🌤️ Weather in {locs[0]}: 28°C, partly cloudy.'
        return 'Which city do you want the weather for?'

    return responses.get(intent, responses['unknown'])


print("Chatbot modules defined: understand() and generate()")

### B2. Testing the Chatbot

In [ ]:
test_inputs = [
    "Hello!",
    "I want to book a flight from Mumbai to Delhi",
    "What's the weather in Pune?",
    "help",
    "Can you fly me from Chennai?",
    "What's the meaning of life?",
    "Bye!"
]

print("=" * 60)
print("CHATBOT TEST RUN")
print("=" * 60)

for user_input in test_inputs:
    nlu = understand(user_input)
    response = generate(nlu)
    print(f"\nYou:  {user_input}")
    print(f"  [NLU] intent={nlu['intent']}, slots={nlu['slots']}")
    print(f"Bot:  {response}")

### B3. Interactive Chat Loop
Run this cell to chat interactively (type `bye` to exit).

In [ ]:
# Uncomment to run interactive chat:
# print("Chatbot ready! Type 'bye' to exit.\n")
# while True:
#     user = input("You: ")
#     nlu = understand(user)
#     if nlu['intent'] == 'farewell':
#         print("Bot:", generate(nlu))
#         break
#     print("Bot:", generate(nlu))

---

## Summary & Observations

| Component | What we learned |
|---|---|
| **Machine Translation** | Used `deep-translator` API for multi-language translation; evaluated quality via BLEU score |
| **NLU** | Regex-based intent detection and slot extraction from user utterances |
| **NLG** | Template-based response generation conditioned on intent + slots |
| **Chatbot Architecture** | Input → NLU → Dialogue Manager → NLG → Output pipeline |